In [0]:
def scalar(q):
    return spark.sql(q).collect()[0][0]

CLEAN = "workspace.oil_gas_silver.telemetry_clean"
QUAR = "workspace.oil_gas_silver.telemetry_quarantine"
BRONZE = "workspace.oil_gas_bronze.telemetry_raw"
GOLD = "workspace.oil_gas_gold.well_daily_summary"

# 1. Reconciliation: nothing lost between Bronze and Silver
clean, quar, bronze = scalar(f"SELECT COUNT(*) FROM {CLEAN}"), scalar(f"SELECT COUNT(*) FROM {QUAR}"), scalar(f"SELECT COUNT(*) FROM {BRONZE}")
assert clean + quar == bronze, f"reconciliation failed: {clean} + {quar} != {bronze}"

# 2. No duplicate keys in Silver
dups = scalar(f"SELECT COUNT(*) FROM (SELECT well_id, reading_ts FROM {CLEAN} GROUP BY well_id, reading_ts HAVING COUNT(*) > 1)")
assert dups == 0, f"{dups} duplicated keys in silver"

# 3. No negative values in Silver
neg = scalar(f"SELECT COUNT(*) FROM {CLEAN} WHERE pressure_psi < 0 OR flow_bpd < 0")
assert neg == 0, f"{neg} negative rows in silver"

# 4. Every quarantined row has a reason
no_reason = scalar(f"SELECT COUNT(*) FROM {QUAR} WHERE reject_reason IS NULL")
assert no_reason == 0, f"{no_reason} quarantine rows without reason"

# 5. Gold reconciles with Silver
gold_total = scalar(f"SELECT SUM(readings_count) FROM {GOLD}")
assert gold_total == clean, f"gold {gold_total} != silver {clean}"

print("all tests passed")